## Import

In [ ]:
# 1) Environment / Imports (for debugging)
import pathlib
import sys

print("python:", sys.version)
print("executable:", sys.executable)

# This repo uses Python 3.10+ typing syntax (e.g. X | Y).
if sys.version_info < (3, 10):
    raise RuntimeError(
        "Python 3.10+ is required for this notebook. "
        "In VS Code: Select Kernel / Python Interpreter -> choose a 3.10+ environment."
    )

import matplotlib.pyplot as plt
import numpy as np

import ansys.motorcad.core as pymotorcad

# Ensure repo root (folder containing 'tools/' or legacy 'tool/') is on sys.path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

# Force reload of local package during iterative edits
import importlib
import tools.motorCAD.pyMCAD as _pyMCAD
importlib.reload(_pyMCAD)

# Import utilities extracted from the notebook
from tools.motorCAD.pyMCAD import (
    get_magnetic_data,
    get_magnetic_data_from_file,
    get_magnetic_timeseries_from_file,
    interactive_magnetic_plot,
    interactive_magnetic_quiver,
    interactive_b_locus_field_plot,
    get_element_loss_fields,
    interactive_loss_fields_plot,
    mcad_default_export_dir,
    mcad_make_temp_txt_path,
    )


# Export settings
DO_EXPORT = True
first_step = 1
final_step = 45

In [ ]:

# Connect to Motor-CAD (uses existing instance by default)
mc = pymotorcad.MotorCAD(open_new_instance=False)

In [ ]:
# (Optional) Auto-reload local modules while you debug edits in tools/motorCAD/pyMCAD/*.py
# NOTE: Do NOT use raw IPython magics (%load_ext ...) inside try/except; they can raise SyntaxError at parse-time.
try:
    ip = get_ipython()  # type: ignore[name-defined]
except Exception:
    ip = None

if ip is None:
    print("autoreload not available (not running in IPython)")
else:
    try:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
        print("autoreload enabled")
    except Exception as e:
        print("autoreload not available:", e)

In [ ]:
# 1.5) Enable zoomable Matplotlib backend (ipympl)
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    import matplotlib
    print("matplotlib backend:", matplotlib.get_backend())
except Exception as e:
    print("ipympl/widget backend not available; using default backend. Error:", e)

In [ ]:
mc.initialise_tab_names()

In [ ]:
FEAscreenName=r'E-Magnetics;FEA'

# % FEAscreenFileName=fullfile(fileDir,[screenFEA{2},'.png']);
mc.display_screen(FEAscreenName)
# mcad.SetVariable('FEShading_Magnetic',1)
# mcad.InitialiseTabNames()
# mcad.SaveMotorCADScreenToFile(FEAscreenName,"Z:\fea.png")

In [ ]:
mc.load_results('Emagnetic')

In [ ]:
mc.load_fea_result(r"E:\KDH\AdaptiveTemplate\TestCAD1\FEResultsData\OnLoadLoss_result_1.mes",1)

In [ ]:
mc.load_fea_result(r"E:\KDH\AdaptiveTemplate\TestCAD1\FEResultsData\OnLoadTorque_result_1.mes",1)

In [ ]:
mc.display_screen("Geometry;Axial")

# 1. Magnetic FEA export & debug
이 노트북은 `tool/motorCAD/pyMCAD` 아래로 분리된 모듈을 import해서 실행/디버깅합니다.

In [ ]:
# 2) Export + Parse (time series)
from pathlib import Path
from datetime import datetime

# Output file base name
out_dir = mcad_default_export_dir(mc)


In [ ]:
base_filename = Path(out_dir) / "MagTransient.txt"
print("Output dir:", out_dir)
print("Base output file:", base_filename)


def _unique_filename_if_exists(path: Path) -> Path:
    """If 'path' exists, return a new path with a timestamp suffix."""
    if not path.exists():
        return path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")


In [ ]:

# Choose filename
if DO_EXPORT:
    filename = _unique_filename_if_exists(base_filename)
    mc.save_fea_data(str(filename), int(first_step), int(final_step), "RegCode,Bx,By,A,J", "", ",")
    print("Exported (unique):", filename)
else:
    filename = base_filename
    print("Skipped export; using existing file:", filename)


In [ ]:
filename=r"D:\KDH\NvidiaNemo\data\MagTransient_20260130_152245.txt"

In [ ]:

# Parse all blocks in file
ts = get_magnetic_timeseries_from_file(filename, key="time_index", verbose=True)
print("Parsed steps:", ts.steps[:10], "..." if len(ts.steps) > 10 else "")

## Plots
아래 셀들은 `ts`(time series)를 대상으로 시각화합니다.

In [ ]:
interactive_magnetic_plot(ts, quantity="b", s=2)

In [ ]:
# Interactive quiver (step toggle)
# Tip: stride를 키우면 더 가볍게 볼 수 있습니다.
interactive_magnetic_quiver(ts, stride=1, scale=1, normalize=False)

# B-locus (all elements)

Plot each element’s $(B_x(t), B_y(t))$ locus as a small loop drawn at that element’s centroid.


In [ ]:
# Interactive B-locus (all elements)
# - Each element’s (Bx(t), By(t)) loop is drawn at its centroid
# - Region dropdown shows "reg_code: region_name" when available
interactive_b_locus_field_plot(ts)

In [ ]:
# Mesh plot example (tri edges)
ts.plot_mesh(step=ts.steps[0] if ts.steps else 0)

# Loss export & plot
Motor-CAD의 `mc.save_fea_data`로 element-wise 손실 성분을 export한 뒤, `Pt/Phys/Pj/Peddy`를 스칼라 필드로 파싱해서 플로팅합니다. (현재 단위는 W/kg로 가정)

In [ ]:
# Export + parse element-wise losses (single snapshot)
# Tip: loss는 보통 step별로 변하지 않는 집계값이라, 마지막 step만 export하는게 안전합니다.
from pathlib import Path

loss_step = int(final_step)

# Export next to the active .mot (same convention as other domains)
out_dir = mcad_default_export_dir(mc)
base_filename = Path(out_dir) / "LossElement.txt"
loss_export_path = _unique_filename_if_exists(base_filename)
print("Loss export path:", loss_export_path)

loss_fields = get_element_loss_fields(
    mc,
    filename=loss_export_path,
    step=loss_step,
    columns=("Pt", "Phys", "Pj", "Peddy"),
    unit="W/kg",
    clean_up=False,  # 파일을 남겨서 내용 확인 가능
 )
print("Parsed loss components:", list(loss_fields.keys()))

In [ ]:
# Interactive plot (component dropdown)
interactive_loss_fields_plot(loss_fields, s=2)

# 2. Mech FEA Export & Debug

In [ ]:
mc.do_mechanical_calculation()
import importlib
from pathlib import Path
import tools.motorCAD.pyMCAD.stress as stress

importlib.reload(stress)  # pick up local edits without restarting kernel
region_names_to_postprocess = ["Rotor"]

# Export file name (domain-named)
out_dir = mcad_default_export_dir(mc)
base_filename = Path(out_dir) / "MechStress.txt"
stress_filename = _unique_filename_if_exists(base_filename)
print("Stress export file:", stress_filename)

stress_regions = stress.get_stress_data(mc, filename=stress_filename, clean_up=False)

# Single figure + dropdown to select region
stress.interactive_mesh_stress_fields_plot(
    stress_regions,
    region_names=region_names_to_postprocess,
    fields=("svm", "sp1", "sp2"),
    show_mesh=True,
    mesh_alpha=0.25,
    colorbar_location="top",
    title_suffix="(before correction)",
)

# 3. Thermal FEA Export & Debug

In [ ]:
# Thermal static export + plot (ThermalStatic*.txt)
from pathlib import Path
from datetime import datetime

import importlib
import tools.motorCAD.pyMCAD.thermal as thermal
importlib.reload(thermal)

def _unique_filename_if_exists(path: Path) -> Path:
    """If 'path' exists, return a new path with a timestamp suffix."""
    if not path.exists():
        return path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")

# Where to save
out_dir = mcad_default_export_dir(mc)
base_filename = Path(out_dir) / "ThermalStatic.txt"
filename = _unique_filename_if_exists(base_filename)
print("Thermal export file:", filename)

# Export + parse (single snapshot)
thermal_regions = thermal.get_thermal_data(
    mc,
    step=1,
    variables="RegCode,T,G,q",
    filename=filename,
    clean_up=False,
 )

# ---- Plot controls ----
# If the 'all' (merged) temperature looks odd, fix the color scale explicitly here.
t_min = 20  # e.g. 20.0
t_max = 150  # e.g. 140.0
field_clim = {
    "t": (t_min, t_max),
    # "g": (None, None),
    # "q": (None, None),
}


# Single figure + dropdown to select region (includes 'all')
thermal.interactive_mesh_thermal_fields_plot(
    thermal_regions,
    fields=("t", "g", "q"),
    show_mesh=True,
    mesh_alpha=0.25,
    # spacing between subplots
    subplot_wspace=0.5,
    # colorbar placement/spacing
    colorbar_location="top",
    colorbar_pad= 0.7,
    field_clim=field_clim,
    title_suffix="(ThermalStatic)",
)

# 4. get Scalar Data & waveform

In [ ]:
from pathlib import Path
from datetime import datetime

base_dir = Path(mc.get_variable("CurrentMotFileDir_MotorLAB"))
mot_path = Path(mc.get_file_name())
mot_stem = mot_path.stem if mot_path.suffix else mot_path.name
# Where MotorLAB typically writes its own outputs
lab_result_dir = base_dir / mot_stem

# Put our exports in a dedicated subfolder (safe, non-destructive)
export_dir = lab_result_dir / "_py_exports"
export_dir.mkdir(parents=True, exist_ok=True)
# Keep variable name for downstream convenience
working_folder = export_dir

mc.show_magnetic_context()

shaft_torque = mc.get_variable("ShaftTorque")
line_voltage = mc.get_variable("PeakLineLineVoltage")

In [ ]:
# Back-EMF scalar metrics (functionized)
from tools.motorCAD.pyMCAD.melec_req_check import get_back_emf_metrics

metrics = get_back_emf_metrics(mc)
for k in ["ShaftTorque", "PeakLineLineVoltage", "PeakBackEMFLine", "RMSBackEMFLine", "PeakBackEMFPhase", "THDBackEMFLine"]:
    if k in metrics:
        unit = ""
        if "Torque" in k:
            unit = " Nm"
        elif "Voltage" in k or "EMF" in k:
            unit = " V"
        elif "THD" in k:
            unit = " %"
        print(f"{k}: {metrics[k]:.4g}{unit}")

In [ ]:
# Back-EMF Waveforms (functionized)
from tools.motorCAD.pyMCAD.melec_req_check import plot_back_emf_waveforms

fig, ax = plot_back_emf_waveforms(mc)

## load

In [ ]:
# TorqueVW waveform (functionized)
import importlib
import tools.motorCAD.pyMCAD.melec_req_check as melec_req_check

importlib.reload(melec_req_check)  # pick up local edits without restarting kernel

In [ ]:
# export_txt_path=None 으로 두면 저장 안함
fig, ax = melec_req_check.plot_torque_vw_waveform(
    mc,
    # export_txt_path=working_folder,  # 폴더를 주면 자동으로 파일명 생성
    export_txt_path=r"E:\\KDH\\out\\TorqueVW.txt",  # 파일을 직접 지정해도 됨
    export_overwrite=False,
    export_delimiter="\t",
    export_include_header=True,
    title="TorqueVW",
    xlabel="Rotor Position",
    ylabel="Electromagnetic Torque [Nm]",
)

In [ ]:
data_type

## Graph Catalog (INI) + Live API Fetch
- `testGraph.ini` is used only to list available graph names and decide `DataType`.
- Actual waveform data is always fetched from Motor-CAD API (`get_magnetic_graph` / `get_fea_graph`).

In [ ]:
from pathlib import Path
import importlib

# Always reload the implementation module so new function arguments are visible
import tools.motorCAD.pyMCAD.melec_req_check as melec_req_check
importlib.reload(melec_req_check)

list_graph_names_from_ini = melec_req_check.list_graph_names_from_ini
get_graph_type_map_from_ini = melec_req_check.get_graph_type_map_from_ini
plot_graph_waveform = melec_req_check.plot_graph_waveform
plot_multi_graph_waveforms = melec_req_check.plot_multi_graph_waveforms

# Baseline catalog INI (static). Adjust to your environment.
ini_path = Path(r"e:\\KDH\\AdaptiveTemplate\\TestCAD1\\testGraph.ini")

type_map = get_graph_type_map_from_ini(ini_path)
print(f"INI graph count: {len(type_map)}")

# Example 1) Terminal voltage (pick first match from catalog, fetch data via API)
tv_names = list_graph_names_from_ini(ini_path, contains=["TerminalVoltage"])
print("Terminal voltage candidates:", tv_names[:10])
if tv_names:
    name = tv_names[0]
    dt = type_map.get(name)
    fig, ax = plot_graph_waveform(
        mc,
        name,
        data_type=dt,
        title=f"{name} ({dt})",
        xlabel="Rotor Position",
        ylabel="Voltage",
    )

# Example 2) Flux linkage (try 3-phase, fetch each trace via API)
fl_names = list_graph_names_from_ini(ini_path, contains=["FluxLinkage"])
print("Flux linkage candidates:", fl_names[:10])
if len(fl_names) >= 3:
    names3 = fl_names[:3]
    fig, ax = plot_multi_graph_waveforms(
        mc,
        names3,
        data_type=type_map.get(names3[0]),
        title="Flux Linkage (3-phase)",
        xlabel="Rotor Position",
        ylabel="Flux Linkage",
        export_txt_path=working_folder,  # 폴더를 주면 자동으로 파일명 생성
        export_format="auto",
        export_delimiter="\t",
    )



# Motor Analysis Process 1-2-3-4 

In [ ]:
Edeg,Fluxlinkage1=mc.get_magnetic_graph(graph_name="FluxLinkageLoadPh1")

In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np

# --- Export settings ---
graph_name = "FluxLinkageLoadPh1"
out_dir = Path(r"E:\\KDH\\out")  # 원하는 폴더로 변경
out_dir.mkdir(parents=True, exist_ok=True)

# --- Fetch data from Motor-CAD ---
Edeg, Fluxlinkage1 = mc.get_magnetic_graph(graph_name=graph_name)

# --- Build output path (avoid overwrite) ---
base = out_dir / f"{graph_name}.txt"
if base.exists():
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_path = base.with_name(f"{base.stem}_{stamp}{base.suffix}")
else:
    out_path = base

# --- Save as 2-column TXT ---
data = np.column_stack([np.asarray(Edeg, dtype=float), np.asarray(Fluxlinkage1, dtype=float)])
np.savetxt(
    out_path,
    data,
    delimiter="\t",
    header="Edeg\tFluxLinkage",
    comments="",
    fmt="%.12g",
)
print("Saved:", out_path)
print("Rows:", data.shape[0])

In [ ]:
fig, ax = plot_multi_graph_waveforms(mc, names3, data_type='FluxLinkageLoadPh1', title="Flux Linkage", export_txt_path=r"E:\KDH\out\flux_linkage.txt")